<!--nav--> [🗺 Learning path](README.md) · **22/46** · ◀ [From Fine-Tune to Production](./From_FineTune_To_Production.ipynb) · [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) ▶

# The Serving Playbook — Start Here

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/The_Serving_Playbook.ipynb)

The notebooks after this one are deep dives. **This one is the front door.** You arrive with a symptom
— "our p95 is bad", "the GPU bill doubled", "it got slow when conversations got long" — and leave
with a diagnosis, a ranked plan, and a pointer to the exact notebook and section that handles it.

| Part | What it does |
|---|---|
| **1** | **Diagnose** — paste your numbers, get your bottleneck and the evidence for it |
| **2** | **Prescribe** — an ordered plan, budgeted by accuracy risk and effort |
| **3** | **Verify** — what to measure afterwards, and what "better" should look like |
| **4** | **The map** — which notebook answers which question |
| **5** | **Self-test** — the track's shared physics, asserted, so you can trust the numbers |
| **6** | **Misdiagnoses** — five symptoms that routinely get blamed on the wrong thing |

**Runs on:** any CPU. Nothing here needs a GPU — it reasons about numbers *you* collect from
[Reading the Logs](./Serving_Logs_Observability.ipynb).

> **The five-minute path:** run Part 1 with your numbers → run Part 2 → open the one notebook it
> points you at. Everything else is reference.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · Diagnose

Fill in what you know. Everything is optional — the diagnosis degrades gracefully and tells you
which missing number would sharpen it. All of these come from
[Reading the Logs](./Serving_Logs_Observability.ipynb): the startup log, the periodic stats line, and
`/metrics`.

In [ ]:
# ── EDIT THIS with your own numbers, then run. Leave anything you don't know as None. ──
OBSERVED = dict(
    # from the periodic stats line (logs Part 2)
    kv_usage_pct      = 94.0,     # "GPU KV cache usage"
    running           = 31,       # "Running: N reqs"
    waiting           = 46,       # "Waiting: N reqs"
    prefix_hit_pct    = 4.0,      # "Prefix cache hit rate"
    prompt_tps        = 5200.0,   # "Avg prompt throughput"
    gen_tps           = 240.0,    # "Avg generation throughput"
    # from /metrics histograms (logs Part 3)
    ttft_p95_s        = 6.4,
    tpot_p95_ms       = 48.0,
    preemptions       = 12,
    # your configuration
    max_num_seqs      = 32,
    max_model_len     = 32768,
    typical_ctx       = 12000,
    # your SLO
    slo_ttft_s        = 1.0,
    slo_tpot_ms       = 50.0,
)

def diagnose(o):
    findings = []
    def add(sev, name, evidence, action, ref):
        findings.append(dict(severity=sev, name=name, evidence=evidence, action=action, ref=ref))

    kv, wait, run = o.get("kv_usage_pct"), o.get("waiting"), o.get("running")
    seqs, hit = o.get("max_num_seqs"), o.get("prefix_hit_pct")
    ptps, gtps = o.get("prompt_tps"), o.get("gen_tps")

    # --- capacity: is memory or the scheduler the binding constraint? ---
    if kv is not None and kv > 90:
        add(3, "KV cache exhausted",
            f"KV usage {kv:.0f}% (>90%)" + (f", {o['preemptions']} preemptions" if o.get("preemptions") else ""),
            "Reduce KV demand before anything else: right-size --max-model-len, enable "
            "--kv-cache-dtype fp8, or add replicas.",
            "the logs notebook Part 4 · the long-context notebook Parts 1-3")
    elif kv is not None and wait and run and seqs and run >= seqs * 0.95 and kv < 70:
        add(2, "Scheduler-bound, not memory-bound",
            f"Running pinned at max_num_seqs={seqs} with {wait} waiting, but KV only {kv:.0f}%",
            "Raise --max-num-seqs. The memory is sitting there unused.",
            "the logs notebook Part 5 · the benchmarking notebook Part 3")

    # --- queueing ---
    if wait is not None and run is not None and (wait > max(1, 0.3 * (run + wait))):
        add(2, "Requests are queueing",
            f"{wait} waiting vs {run} running - queue time is inside every user's TTFT",
            "You are past the knee. Either shed load (bounded queue + 429) or add capacity.",
            "the benchmarking notebook Parts 3-4 · the hardening notebook Part 2")

    # --- workload shape ---
    if ptps and gtps and ptps > 5 * gtps:
        add(2, "Prefill-dominated workload",
            f"prefill {ptps:,.0f} tok/s vs decode {gtps:,.0f} tok/s (>5x)",
            "Long prompts are starving decode. Check chunked prefill, and attack the prompts "
            "themselves - this shape is usually RAG or agent traffic.",
            "the long-context notebook Part 5 · the RAG & agents notebook Parts 1-3")

    if hit is not None and hit < 15 and ptps and gtps and ptps > gtps:
        add(3, "Prefix cache is cold on repetitive traffic",
            f"prefix hit rate {hit:.0f}% despite a prefill-heavy workload",
            "Almost always a prompt-layout bug: something volatile (timestamp, request id, "
            "the user's question) sits near the front. Reorder static-first.",
            "the RAG & agents notebook Part 2 · the vLLM notebook Part 4")

    # --- SLO ---
    if o.get("ttft_p95_s") and o.get("slo_ttft_s") and o["ttft_p95_s"] > o["slo_ttft_s"]:
        add(2, "TTFT SLO breached",
            f"p95 TTFT {o['ttft_p95_s']:.1f}s vs SLO {o['slo_ttft_s']:.1f}s",
            "TTFT is prefill + queue time. Fix the queue first (it is usually the bigger half), "
            "then prefill via caching.",
            "the serving-fundamentals notebook Part 2 · the benchmarking notebook Part 3")
    if o.get("tpot_p95_ms") and o.get("slo_tpot_ms") and o["tpot_p95_ms"] > o["slo_tpot_ms"]:
        add(2, "TPOT SLO breached",
            f"p95 TPOT {o['tpot_p95_ms']:.0f}ms vs SLO {o['slo_tpot_ms']:.0f}ms",
            "Per-token latency is the decode step. Find your biggest slice before choosing a fix.",
            "the decode-anatomy notebook Parts 3-5")

    # --- configuration smells ---
    if o.get("max_model_len") and o.get("typical_ctx") and o["max_model_len"] > 3 * o["typical_ctx"]:
        add(1, "max-model-len far above real usage",
            f"--max-model-len {o['max_model_len']:,} but typical context ~{o['typical_ctx']:,}",
            "Not directly harmful, but it caps concurrency planning and inflates worst-case "
            "reservations. Right-size it and re-read the startup log.",
            "the logs notebook Part 1")

    return sorted(findings, key=lambda f: -f["severity"])

findings = diagnose(OBSERVED)
SEV = {3: "🔴 CRITICAL", 2: "🟠 IMPORTANT", 1: "🟡 MINOR"}
print(f"{len(findings)} finding(s), most severe first:\n")
for f in findings:
    print(f"{SEV[f['severity']]}  {f['name']}")
    print(f"     evidence : {f['evidence']}")
    print(f"     action   : {f['action']}")
    print(f"     read     : {f['ref']}\n")
if not findings:
    print("Nothing alarming. If you still want more headroom, go to Part 2 and optimize "
          "deliberately rather than reactively.")

**The example above is a real composite** — KV exhausted, queue building, prefix cache cold on
prefill-heavy traffic. That combination is the single most common production shape for RAG and agent
products, and note that the three findings have **one root cause**: enormous prompts that aren't
being reused. Fixing the prompt layout addresses all three; adding GPUs addresses none of them
permanently.

## Part 2 · Prescribe

Now the ordered plan. This reuses the composition model from
[The Optimization Stack](./The_Optimization_Stack.ipynb) — so the ordering accounts for the fact that
optimizations interact rather than multiply.

In [ ]:
# The optimization catalogue, with what it needs to be true in order to pay off.
PLAYS = [
    dict(name="Fix prompt layout (static → volatile)",
         gate=lambda o: (o.get("prefix_hit_pct") is not None and o["prefix_hit_pct"] < 40),
         gain="1.5-20x on prefill for repetitive traffic", risk=0, effort=1,
         ref="the RAG & agents notebook Part 2"),
    dict(name="Enable / verify prefix caching",
         gate=lambda o: (o.get("prefix_hit_pct") is not None and o["prefix_hit_pct"] < 40),
         gain="large on repeated prefixes", risk=0, effort=1, ref="the vLLM notebook Part 4"),
    dict(name="Prefix-aware or session-affinity routing",
         gate=lambda o: (o.get("prefix_hit_pct") is not None and o["prefix_hit_pct"] < 40),
         gain="makes the cache actually hit", risk=0, effort=2, ref="the distributed-serving notebook Part 4"),
    dict(name="Right-size --max-model-len",
         gate=lambda o: (o.get("max_model_len") and o.get("typical_ctx")
                         and o["max_model_len"] > 3 * o["typical_ctx"]),
         gain="more KV pool, higher concurrency", risk=0, effort=1, ref="the logs notebook Part 1"),
    dict(name="Raise --max-num-seqs",
         gate=lambda o: (o.get("kv_usage_pct") is not None and o["kv_usage_pct"] < 70
                         and o.get("waiting", 0) > 0),
         gain="uses memory you already paid for", risk=0, effort=1, ref="the benchmarking notebook Part 3"),
    dict(name="Enable CUDA/HIP graph capture",
         gate=lambda o: True,
         gain="removes fixed per-step overhead (big at low batch)", risk=0, effort=1,
         ref="the decode-anatomy notebook Part 4"),
    dict(name="FP8 KV cache",
         gate=lambda o: (o.get("kv_usage_pct") or 0) > 70,
         gain="~2x concurrent sequences", risk=1, effort=1, ref="the long-context notebook Part 3"),
    dict(name="Quantize weights (AWQ/GPTQ/FP8)",
         gate=lambda o: True,
         gain="1.3-2x decode when memory-bound", risk=2, effort=1, ref="the quantization notebook"),
    dict(name="Speculative or n-gram decoding",
         gate=lambda o: (o.get("running") or 0) < (o.get("max_num_seqs") or 999) * 0.7,
         gain="1.5-3x at low/medium load, provably lossless", risk=0, effort=3, ref="the speculation notebook"),
    dict(name="Bounded queue + 429 with Retry-After",
         gate=lambda o: (o.get("waiting") or 0) > 5,
         gain="converts invisible waste into actionable rejections", risk=0, effort=2,
         ref="the hardening notebook Part 2"),
    dict(name="Add replicas (with prefix-aware routing)",
         gate=lambda o: (o.get("kv_usage_pct") or 0) > 90 or (o.get("waiting") or 0) > 20,
         gain="linear, and the only real fix once tuned out", risk=0, effort=3,
         ref="the distributed-serving notebook · the benchmarking notebook Part 5"),
    dict(name="KV eviction / sliding-window attention",
         gate=lambda o: (o.get("typical_ctx") or 0) > 16000,
         gain="large KV reduction at long context", risk=4, effort=3, ref="the long-context notebook Part 4"),
]

def prescribe(o, max_risk=2):
    applicable = [p for p in PLAYS if p["gate"](o)]
    ordered = sorted(applicable, key=lambda p: (p["risk"], p["effort"]))
    return [p for p in ordered if p["risk"] <= max_risk], \
           [p for p in ordered if p["risk"] > max_risk]

plan, deferred = prescribe(OBSERVED, max_risk=2)
print(f"Ordered plan ({len(plan)} plays) — cheapest and safest first:\n")
print(f"{'#':>3}  {'play':<44}{'risk':>5}{'effort':>8}   read")
print("-" * 92)
for i, p in enumerate(plan, 1):
    print(f"{i:>3}. {p['name']:<44}{p['risk']:>5}{p['effort']:>8}   {p['ref']}")
    print(f"     expected: {p['gain']}")

if deferred:
    print(f"\nDeferred (accuracy risk above your budget):")
    for p in deferred:
        print(f"     {p['name']:<44}risk {p['risk']}   {p['ref']}")

print("\nWork DOWN this list, and re-measure after each step (Part 3).")
print("The order is not cosmetic: every zero-risk play comes before every lossy one, and")
print("each one you complete changes what the next is worth (optimization-stack).")

## Part 3 · Verify

After each change, confirm the mechanism actually engaged — not just that the dashboard looks nicer.
**Most failed optimizations are optimizations that never took effect.**

| You changed | The metric that must move | If it doesn't |
|---|---|---|
| Prompt layout | `prefix cache hit rate` ↑ | something volatile is still near the front ([RAG & Agent Serving Patterns](./RAG_Agent_Serving_Patterns.ipynb)) |
| `--kv-cache-dtype fp8` | `gpu_cache_usage_perc` ↓ at equal load | the flag isn't supported on your GPU ([Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb)) |
| Quantized weights | startup log "Model loading took … GiB" ↓ | you loaded the unquantized checkpoint ([Reading the Logs](./Serving_Logs_Observability.ipynb)) |
| `--max-num-seqs` ↑ | `Running` ↑, `Waiting` ↓ | you were KV-bound, not slot-bound |
| Speculative decoding | acceptance rate visible & > ~0.6 | bad draft model; may be slowing you down ([Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb)) |
| Added replicas | per-replica `Running` ↓ | the router isn't spreading load ([Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb)) |
| Bounded queue | 429s appear; p95 TTFT ↓ | limit is still too high ([Production Hardening](./Production_Hardening_Reliability.ipynb)) |

**And always re-run the benchmark** ([Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb)),
open-loop, at your real prompt distribution. A dashboard improving under different traffic is not
evidence.

In [ ]:
# A tiny before/after checker you can keep: did the change do what it claimed?
EXPECTED_EFFECT = {
    "prompt layout":        ("prefix_hit_pct",   "up"),
    "fp8 kv cache":         ("kv_usage_pct",     "down"),
    "raise max-num-seqs":   ("waiting",          "down"),
    "quantize weights":     ("tpot_p95_ms",      "down"),
    "add replicas":         ("waiting",          "down"),
    "bounded queue":        ("ttft_p95_s",       "down"),
}

def verify(change, before, after, min_rel_change=0.10):
    if change not in EXPECTED_EFFECT:
        return f"no expectation registered for '{change}'"
    metric, direction = EXPECTED_EFFECT[change]
    b, a = before.get(metric), after.get(metric)
    if b is None or a is None:
        return f"⚠ cannot verify: '{metric}' missing from one of the samples"
    rel = (a - b) / abs(b) if b else 0
    moved = (rel > min_rel_change) if direction == "up" else (rel < -min_rel_change)
    arrow = "↑" if rel > 0 else "↓"
    return (f"{'✅' if moved else '❌'} {change}: {metric} {b} → {a} ({arrow}{abs(rel):.0%}) "
            f"— expected {direction}" + ("" if moved else "  ⇒ THE CHANGE DID NOT TAKE EFFECT"))

before = dict(OBSERVED)
after_good = {**OBSERVED, "prefix_hit_pct": 68.0, "kv_usage_pct": 61.0,
              "waiting": 2, "ttft_p95_s": 0.7}
after_noop = {**OBSERVED, "prefix_hit_pct": 4.2}

print("A change that worked:")
print("  " + verify("prompt layout", before, after_good))
print("  " + verify("fp8 kv cache", before, after_good))
print("  " + verify("bounded queue", before, after_good))
print("\nA change that silently did nothing (the common case):")
print("  " + verify("prompt layout", before, after_noop))

## Part 4 · The map

Which notebook answers which question:

| Your question | Go to |
|---|---|
| Why is decode slow no matter what I do? | [21 Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb), [31 Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) |
| What is my engine actually doing? | [22 vLLM](./vLLM_High_Throughput_Serving.ipynb), [25 Visualized](./Serving_Internals_Visualized_D3.ipynb) |
| Where do the milliseconds go? | [38 Anatomy of a Decode Step](./Anatomy_Of_A_Decode_Step.ipynb) |
| Should I quantize, and to what? | [23 Quantized Showdown](./Quantized_Serving_Showdown.ipynb), [32 Precision Matrix](./Portable_Kernels_Precision_Matrix.ipynb) |
| Can I make single-user latency better? | [24 Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb) |
| How do I read the logs / what should I alert on? | [26 Observability](./Serving_Logs_Observability.ipynb) |
| How many GPUs do I need, and what will it cost? | [27 Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb), [33 What-If Console](./Serving_WhatIf_Console.ipynb) |
| How do I guarantee valid JSON / tool calls? | [28 Structured Output](./Structured_Output_Guided_Decoding.ipynb) |
| One big model or many replicas? | [29 Distributed Serving](./Distributed_MultiReplica_Serving.ipynb) |
| Many customers, many fine-tunes | [30 Multi-LoRA](./MultiLoRA_Serving_At_Scale.ipynb) |
| NVIDIA or AMD? Will my config port? | [31 Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb), [32 Precision Matrix](./Portable_Kernels_Precision_Matrix.ipynb) |
| It broke at 128k context | [34 Long Context](./LongContext_KV_Compression_Serving.ipynb) |
| I'm serving a Mixture-of-Experts model | [35 MoE](./MoE_Serving_Expert_Parallelism.ipynb) |
| My agent got slower every turn | [36 RAG & Agents](./RAG_Agent_Serving_Patterns.ipynb) |
| How do I not get paged at 3am? | [37 Production Hardening](./Production_Hardening_Reliability.ipynb) |
| I applied five optimizations and got 2× | [39 The Optimization Stack](./The_Optimization_Stack.ipynb) |

## Part 5 · Self-test

A playbook is only as trustworthy as the physics under it. These assertions restate the track's core
relationships and check them numerically — if any fails, something in the reasoning above is wrong
and you should stop trusting it.

In [ ]:
def check(name, got, want, tol=0.02, unit=""):
    ok = abs(got - want) <= tol * max(abs(want), 1e-12)
    print(f"{'✅' if ok else '❌'} {name:<52}{got:>12,.3f}{unit}  (expected {want:,.3f}{unit})")
    return ok

results = []
print("Core relationships from across the track:\n")

# the serving-fundamentals notebook: KV bytes/token for Llama-3.1-8B GQA in fp16
kv_per_token = 2 * 32 * 8 * 128 * 2
results.append(check("KV bytes/token, 8B GQA fp16 (serving-fundamentals)", kv_per_token / 1024, 128.0, unit=" KB"))

# serving-fundamentals, roofline: decode is memory-bound => tok/s = bandwidth / model bytes
bw, model_bytes = 3.35e12 * 0.75, 8e9 * 2
results.append(check("8B fp16 decode ceiling, H100, batch 1 (the roofline notebook)",
                     bw / model_bytes, 157.0, tol=0.05, unit=" tok/s"))

# the roofline notebook: ridge point = peak FLOPS / bandwidth; in fp16 the flip batch equals it
ridge = 990e12 / 3.35e12
results.append(check("H100 ridge point (roofline)", ridge, 295.5, tol=0.02, unit=" FLOP/B"))
results.append(check("  => fp16 flip batch equals the ridge", ridge * 2 / 2, ridge))
results.append(check("  => int4 flip batch is 4x smaller", ridge * 0.5 / 2, ridge / 4))

# the speculation notebook: expected tokens per verify round
a, k = 0.75, 4
results.append(check("speculation E[tokens/round], a=0.75 k=4 (the speculation notebook)",
                     (1 - a ** (k + 1)) / (1 - a), 3.051, tol=0.01))

# the distributed-serving notebook: TP=4 efficiency at 6% communication
eff = (1 / (1 / 4 + 0.06 * 3 / 4)) / 4
results.append(check("TP=4 efficiency at 6% comm (distributed-serving)", eff, 0.847, tol=0.01))

# the MoE notebook: MoE arithmetic intensity is B*k/E, not B
results.append(check("MoE AI at batch 32, top-8, 256 experts (the MoE notebook)",
                     2 * max(1.0, 32 * 8 / 256) / 2, 1.0, tol=0.01, unit=" FLOP/B"))

# the benchmarking notebook: Little's law - queue wait explodes as utilisation approaches 1
util = 0.9
results.append(check("M/M/1 wait multiplier at 90% utilisation (the benchmarking notebook)",
                     1 / (1 - util), 10.0, tol=0.01, unit="x"))

# the RAG & agents notebook: agent prefill is quadratic without caching
n, base, per = 20, 2000, 400
no_cache = sum(base + per * t for t in range(n))
cached = base + per * n
results.append(check("20-turn agent prefill reduction from caching (the RAG & agents notebook)",
                     1 - cached / no_cache, 0.914, tol=0.02))

print(f"\n{sum(results)}/{len(results)} invariants hold.")
if all(results):
    print("The physics underneath this playbook is internally consistent.")
else:
    print("⚠ A relationship above does not hold - treat the prescriptions with suspicion.")

print("\n(The repo also ships `python tools/audit_consistency.py`, which cross-checks the")
print(" hardware and model constants shared between notebooks so they cannot silently drift.)")

## Part 6 · Five misdiagnoses

Symptoms that get blamed on the wrong thing, over and over:

| Symptom | Usual blame | Actually, usually | Check |
|---|---|---|---|
| "Latency is bad, we need faster GPUs" | compute | **queueing** — you're past the knee, so wait time dominates | `num_requests_waiting` ([Reading the Logs](./Serving_Logs_Observability.ipynb)) |
| "We enabled prefix caching and nothing happened" | the feature | **prompt layout** — a timestamp or ID near the front | hit rate ([RAG & Agent Serving Patterns](./RAG_Agent_Serving_Patterns.ipynb) Part 2) |
| "We quantized and throughput barely moved" | quantization | **you were already compute-bound**, or attention dominates | [Anatomy of a Decode Step](./Anatomy_Of_A_Decode_Step.ipynb) Part 3 |
| "Speculative decoding made it slower" | the technique | **too little spare compute at your batch size** | acceptance rate + batch ([Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb), [The Optimization Stack](./The_Optimization_Stack.ipynb)) |
| "It got slow as conversations got longer" | the model | **quadratic prefill** from re-sending history | [RAG & Agent Serving Patterns](./RAG_Agent_Serving_Patterns.ipynb) Part 1 |

The pattern behind all five: **the symptom is at the end of a causal chain, and the fix is at the
beginning.** That's the same lesson [Reading the Logs](./Serving_Logs_Observability.ipynb) drew from its incident timeline — alert on causes,
not symptoms.

## Where to go now

- Just arrived? Run Part 1 with your numbers, then follow Part 2's first play.
- Want the mental model rather than a fix? [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) →
  [Serving Internals Visualized](./Serving_Internals_Visualized_D3.ipynb) → [Anatomy of a Decode Step](./Anatomy_Of_A_Decode_Step.ipynb).
- Choosing hardware? [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) →
  [The What-If Console](./Serving_WhatIf_Console.ipynb).
- About to stack several optimizations? Read [The Optimization Stack](./The_Optimization_Stack.ipynb) **first** — it will
  change the order you do them in.

🏁 [Back to the learning path](README.md)